# 06 Credit Risk Assessment

Eksperimen memprediksi risiko gagal bayar (loan default) nasabah menggunakan dataset Excel multi-sheet.

## Tahapan Eksperimen
1. Read & Merge dataset Excel (`Data_Debitur` dan `Riwayat_Pinjaman`).
2. Cleaning missing values & feature engineering (rasio angsuran).
3. Modeling dengan Random Forest (`class_weight="balanced"`).
4. Evaluasi metrik (Confusion Matrix, ROC-AUC, PR-AUC, dan Feature Importance).

## 1. Import Library & Data Loading (Excel Multi-Sheet)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, average_precision_score

# Baca data Excel multi-sheet
excel_path = "dataset_kredit.xlsx"

sheet_debitur = pd.read_excel(excel_path, sheet_name="Data_Debitur")
sheet_pinjaman = pd.read_excel(excel_path, sheet_name="Riwayat_Pinjaman")

print("Data Debitur:")
display(sheet_debitur.head())

print("Riwayat Pinjaman:")
display(sheet_pinjaman.head())

# Merge berdasarkan Primary Key id_nasabah
df = pd.merge(sheet_debitur, sheet_pinjaman, on="id_nasabah")
print(f"Total data setelah merge: {df.shape[0]} baris, {df.shape[1]} kolom")

## 2. Pembersihan Data & Feature Engineering

In [ ]:
# Imputasi Missing Values
df["status_pekerjaan"] = df["status_pekerjaan"].fillna(df["status_pekerjaan"].mode()[0])
df["skor_kredit_internal"] = df["skor_kredit_internal"].fillna(df["skor_kredit_internal"].median())

# Feature Engineering Finansial
df["angsuran_estimasi"] = (df["plafon_pinjaman"] / df["tenor_bulan"]) * 1.10
df["rasio_angsuran_pendapatan"] = np.round(df["angsuran_estimasi"] / df["pendapatan_bulanan"], 4)

# Encoding Variabel Kategorikal
df_encoded = pd.get_dummies(df, columns=["status_pekerjaan"], drop_first=True)
df_encoded.head()

## 3. Pelatihan Model Random Forest dengan Balanced Class Weight

In [ ]:
feature_cols = [
    "usia", "pendapatan_bulanan", "jumlah_tanggungan", "skor_kredit_internal",
    "plafon_pinjaman", "tenor_bulan", "rasio_dti", "jumlah_keterlambatan",
    "angsuran_estimasi", "rasio_angsuran_pendapatan",
    "status_pekerjaan_Karyawan Swasta", "status_pekerjaan_PNS", "status_pekerjaan_Wiraswasta"
]
feature_cols = [c for c in feature_cols if c in df_encoded.columns]

X = df_encoded[feature_cols]
y = df_encoded["status_gagal_bayar"]

# Train/Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Model Training
rf = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC Score : {average_precision_score(y_test, y_prob):.4f}")

## 4. Analisis Feature Importance (Interpretabilitas Bisnis)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.tail(8).plot(kind="barh", color="teal")
plt.title("Top Faktor Kritis Risiko Gagal Bayar Kredit")
plt.xlabel("Tingkat Kepentingan Fitur (Feature Importance)")
plt.tight_layout()
plt.show()